# 02. Benchmark 与轨迹评估

SWE-bench、GAIA、WebArena——这些 benchmark 定义了 Agent 能力的度量标准。理解它们的任务格式、评估方法和局限性，是使用和设计 benchmark 的前提。

> **检查点**：能说出 SWE-bench 和 GAIA 的任务差异（代码修复 vs 多步推理）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 轨迹回放与评估
import json

# 模拟一段 Agent 轨迹
sample_trajectory = [
    {"step":1,"thought":"Need to find the bug","action":"read_file('main.py')","observation":"def add(a,b): return a-b"},
    {"step":2,"thought":"Found: should be a+b","action":"edit_file('main.py','a-b','a+b')","observation":"File edited"},
    {"step":3,"thought":"Verify fix","action":"run_tests()","observation":"2/2 tests passed"},
    {"step":4,"thought":"Done","action":"submit","observation":"Patch submitted"},
]

# Gold standard（理想轨迹）
gold = [
    {"action_type":"read","valid":True},
    {"action_type":"edit","valid":True,"target":"a+b"},
    {"action_type":"test","valid":True},
    {"action_type":"submit","valid":True},
]

def evaluate_trajectory(trace: list, gold: list) -> dict:
    metrics = {
        "step_count": len(trace),
        "gold_step_count": len(gold),
        "step_efficiency": len(gold) / max(len(trace), 1),
        "valid_actions": sum(1 for s in trace if "error" not in s.get("observation","").lower()),
        "success": trace[-1].get("observation","").startswith("Patch") if trace else False,
    }
    metrics["score"] = (
        metrics["step_efficiency"] * 0.3 +
        (metrics["valid_actions"]/max(len(trace),1)) * 0.3 +
        (1.0 if metrics["success"] else 0.0) * 0.4
    )
    return metrics

metrics = evaluate_trajectory(sample_trajectory, gold)
print("Trajectory Evaluation:")
for k, v in metrics.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

Trajectory Evaluation:
  step_count: 4
  gold_step_count: 4
  step_efficiency: 1.00
  valid_actions: 4
  success: True
  score: 1.00


## 主要 Benchmark 对比

| Benchmark | 任务类型 | 评估方式 | 难度 |
|-----------|----------|----------|------|
| SWE-bench Verified | 修复真实 GitHub issue | Patch + 测试 | 高 |
| GAIA | 多步推理（搜索+计算+验证） | 精确匹配 | 中-高 |
| WebArena | Web 任务（导航+填写+提取） | 功能正确性 | 中 |
| AgentBench | 8 种环境（OS/DB/Web/游戏） | 任务完成率 | 中 |

> **检查点**：能为你自己的 Agent 选择合适的 benchmark，或者设计一个最小可用的评估集。

## 真实 API 实验

轨迹回放 + LLM 逐步评审（`trajectory_replay.py --use-api`）。


In [3]:
SCRIPT = ROOT / 'agents' / 'evaluation' / 'scripts' / 'trajectory_replay.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--use-api'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


(no file given — replaying built-in sample trajectory)
Replayed 4 steps:
  step 1: search({"query": "python"})
  step 2: search({"query": "python"}) [REDUNDANT]
  step 3: search({"query": "python history"})
  step 4: done({})

Metrics: valid=100% efficient=75% repeated=1

=== LLM step-by-step review ===
## 逐步点评

- **step 1**：查询“python”得到10个结果，合理但泛泛，可作为初始探索。  
- **step 2**：完全重复上一步，无任何新信息或策略调整，纯属浪费。**最差一步**，直接导致无效开销。  
- **step 3**：终于意识到要细化，改为“python history”得到5个结果，方向正确但为时已晚。  
- **step 4**：以空答案结束，未使用任何搜索结果，前功尽弃，严重失职。

## 总体评分

**1/10**  
原因：重复操作、缺乏思考、最终无输出，完全不可接受。



## 练习

1. 为 `mini_coding_agent.py` 设计 3 个 E2E 测试用例（不同 bug 类型）。计算 pass rate。
2. 实现 `compare_trajectories(trace_a, trace_b)`：对比两条轨迹，找出 Agent 决策的差异点。
3. 思考：一个好的 benchmark 应该具备哪些特性？（提示：难度分层、可重复、防作弊）